## Data Collection

This notebook retrieves historical daily price data for the selected asset
universe and benchmark using the Yahoo Finance API (`yfinance`).

The goal of this step is to:
- Acquire clean, adjusted price series
- Validate data completeness
- Store raw data for downstream processing

No transformations or return calculations are performed here.

In [1]:
import yfinance as yf
import pandas as pd
from pathlib import Path

In [2]:
# Configuration
TICKERS = ["NVDA", "MSFT", "JNJ", "JPM", "RTX", "V"]
BENCHMARK = "^GSPC"
START_DATE = "2014-01-01"
END_DATE = "2024-12-31"

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_prices(ticker):
    df = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=False
    )
    df.reset_index(inplace=True)
    return df

In [ ]:
# Download assets
for ticker in TICKERS + [BENCHMARK]:
    df = download_prices(ticker)
    df.to_csv(RAW_DIR / f"{ticker}_raw.csv", index=False)
    print(f"Saved {ticker} ({len(df)} rows)")

In [3]:
for ticker in TICKERS + [BENCHMARK]:
    df = pd.read_csv(RAW_DIR / f"{ticker}_raw.csv")

    # make Date consistent
    if "Date" not in df.columns:
        # common case: date was saved as index -> comes back as "Unnamed: 0"
        if "Unnamed: 0" in df.columns:
            df = df.rename(columns={"Unnamed: 0": "Date"})
        elif "Datetime" in df.columns:
            df = df.rename(columns={"Datetime": "Date"})
        else:
            raise KeyError(f"{ticker}: couldn't find a Date column. Columns: {list(df.columns)}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    print(
        ticker,
        "| rows:", len(df),
        "| start:", df["Date"].min(),
        "| end:", df["Date"].max(),
        "| bad_dates:", df["Date"].isna().sum()
    )

NVDA | rows: 2768 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00 | bad_dates: 1
MSFT | rows: 2768 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00 | bad_dates: 1
JNJ | rows: 2768 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00 | bad_dates: 1
JPM | rows: 2768 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00 | bad_dates: 1
RTX | rows: 2768 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00 | bad_dates: 1
V | rows: 2768 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00 | bad_dates: 1
^GSPC | rows: 2768 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00 | bad_dates: 1


In [4]:
#Finding our bad_dates
ticker = "NVDA"
df = pd.read_csv(RAW_DIR / f"{ticker}_raw.csv")

# normalize date column name
if "Date" not in df.columns:
    if "Unnamed: 0" in df.columns:
        df = df.rename(columns={"Unnamed: 0": "Date"})
    elif "Datetime" in df.columns:
        df = df.rename(columns={"Datetime": "Date"})
    else:
        raise KeyError(f"{ticker}: no Date-like column. Columns: {list(df.columns)}")

df["Date_parsed"] = pd.to_datetime(df["Date"], errors="coerce")

bad = df[df["Date_parsed"].isna()]
bad[["Date"]].head(10), bad.index[:10]

(  Date
 0  NaN,
 Index([0], dtype='int64'))

In [5]:
df.loc[bad.index, :].head(5)

,Date,Adj Close,Close,High,Low,Open,Volume,Date_parsed
0,NaN,NVDA,NVDA,NVDA,NVDA,NVDA,NVDA,NaT


In [6]:
for ticker in TICKERS + [BENCHMARK]:
    df = pd.read_csv(RAW_DIR / f"{ticker}_raw.csv")

    if "Date" not in df.columns:
        if "Unnamed: 0" in df.columns:
            df = df.rename(columns={"Unnamed: 0": "Date"})
        elif "Datetime" in df.columns:
            df = df.rename(columns={"Datetime": "Date"})
        else:
            raise KeyError(f"{ticker}: couldn't find a Date column. Columns: {list(df.columns)}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    print(ticker, "| rows:", len(df), "| start:", df["Date"].min(), "| end:", df["Date"].max())

NVDA | rows: 2767 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00
MSFT | rows: 2767 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00
JNJ | rows: 2767 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00
JPM | rows: 2767 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00
RTX | rows: 2767 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00
V | rows: 2767 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00
^GSPC | rows: 2767 | start: 2014-01-02 00:00:00 | end: 2024-12-30 00:00:00
